# Phase 1: download each account's website

Reads the accounts in the input Excel and saves each website's pages (text and image labels) in the crawl store,
one line per account. Phase 2 reads only the crawl store.

**How it works** (`crawl.py`): the websites are split between several worker processes (`crawl_worker.py`), in up to
three passes: a fast plain download for every site, a real browser for the sites that refused or gave back next to
nothing, and a last try with an "undetected" browser (Chrome windows open) for those still refused. Each site's home
page comes first, then its most useful pages (`page_rules.py`), up to 40 pages (20 in the browser) and 150 seconds.

Stopping and re-running is always safe: saved accounts are skipped, and sites that refused us or didn't answer are
tried again on later runs (3 tries in all).

In [ ]:
# First time on a new machine only:
# !pip install -U crawl4ai pandas openpyxl requests model2vec python-dotenv
# !crawl4ai-setup

## Settings

In [ ]:
from pathlib import Path
import crawl

INPUT   = Path("data/zoominfo_100each_2.xlsx")          # the accounts: a CRM export (account_id, name, company_website, ...)
STORE   = Path("data/crawls_zoominfo_100each_2.jsonl")  # the crawl store this run writes to
SUMMARY = Path("data/crawl_summary_zoominfo_100each_2.xlsx")

LIMIT       = None  # crawl at most this many accounts this run (e.g. 5 to try it out); None = all that need it
RECRAWL_IDS = []    # account_ids to crawl again even though they are stored

## A new set of accounts (optional)
Writes a new input file of random accounts with a website, leaving out every account used before, so sets never
overlap. `data/zoominfo_100each_2.xlsx` was made this way: 118 per segment label (about 100 are left once dead,
parked and wrong websites drop out).

In [ ]:
# SEGMENTS = ["Decorated Apparel", "Landscape/Hardscape", "Light Construction", "Septic", "Towing"]
# USED_BEFORE = ["data/crawls_tuning.jsonl", "data/crawls_zoominfo_eval.jsonl", "data/crawls_zoominfo_eval2.jsonl",
#                "data/crawls_zoominfo_500.jsonl", "data/crawls_zoominfo_100each.jsonl", "data/zoominfo_100each.xlsx",
#                "data/compare_20each.xlsx"]   # crawl stores or input files whose accounts must not be picked
# crawl.make_sample("data/Dim_Acc_ZI_No_Opp.xlsx", 118, INPUT, exclude=USED_BEFORE, per_label=True, labels=SEGMENTS)

## Run

In [ ]:
crawl.run(INPUT, STORE, limit=LIMIT, recrawl_ids=RECRAWL_IDS)

## How the crawl went
| status | meaning |
|---|---|
| `ok` | pages saved |
| `parked`, `dead_site` | the domain is for sale, or the site no longer exists |
| `blocked`, `unreachable` | it refused us or didn't answer (tried again on the next run) |
| `network_blocked` | our own network filter stopped it |
| `no_website`, `social_media_page_only` | nothing to crawl |
| `wrong_site` | set by Phase 2: not the company's website (crawled again once the CRM has a new URL) |

In [ ]:
summary = crawl.summary(STORE, SUMMARY)
print(summary["status"].value_counts().to_string())
print("\nMedian pages per usable site:", summary.loc[summary.status == "ok", "pages"].median())
summary.head(20)

## Tidy the store (optional)
Re-crawls and website checks add lines; this keeps one line per account.

In [ ]:
# crawl.compact(STORE)